# Attention Weights vs. Integrated Gradients: Empirical Comparison

**Learning objectives:**
- Extract and visualize raw attention weights from all 12 BERT heads
- Compute attention rollout across all layers
- Compare attention vs. IG token rankings on the same sentences
- Demonstrate disagreement cases: negation, adversarial tokens
- Compute Spearman rank correlation between attention and IG

**Estimated time:** 15 minutes

**Model:** `textattack/bert-base-uncased-SST-2` (same as notebook 01)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoModelForSequenceClassification, AutoTokenizer
from captum.attr import LayerIntegratedGradients

# Load model (same as notebook 01)
model_name = "textattack/bert-base-uncased-SST-2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

LABELS = ["NEGATIVE", "POSITIVE"]
print("Model loaded.")

## 1. Helper Functions (from Notebook 01)

In [ ]:
def tokenize(text, max_length=128):
    inputs = tokenizer(text, return_tensors='pt', max_length=max_length,
                       truncation=True, padding=False)
    return inputs['input_ids'], inputs.get('attention_mask'), inputs.get('token_type_ids')

def make_baseline(input_ids):
    return torch.full_like(input_ids, tokenizer.pad_token_id)

def forward_func(input_ids, attention_mask=None, token_type_ids=None):
    return model(input_ids=input_ids, attention_mask=attention_mask,
                 token_type_ids=token_type_ids).logits

def aggregate_subwords(tokens, attrs, skip_special=True):
    special = {'[CLS]', '[SEP]', '[PAD]'}
    word_tokens, word_attrs = [], []
    cur_word, cur_attr = None, 0.0
    for tok, attr in zip(tokens, attrs):
        if skip_special and tok in special: continue
        if tok.startswith('##'):
            cur_word = (cur_word or '') + tok[2:]
            cur_attr += attr
        else:
            if cur_word: word_tokens.append(cur_word); word_attrs.append(cur_attr)
            cur_word, cur_attr = tok, attr
    if cur_word: word_tokens.append(cur_word); word_attrs.append(cur_attr)
    return word_tokens, np.array(word_attrs)

def compute_ig_attrs(text, n_steps=40):
    ids, mask, ttype = tokenize(text)
    baseline = make_baseline(ids)
    with torch.no_grad():
        pred = forward_func(ids, mask, ttype).argmax(dim=1).item()
    lig = LayerIntegratedGradients(forward_func, model.bert.embeddings)
    attrs, delta = lig.attribute(
        ids, baseline, additional_forward_args=(mask, ttype),
        target=pred, n_steps=n_steps, return_convergence_delta=True,
    )
    tokens = tokenizer.convert_ids_to_tokens(ids[0].tolist())
    return tokens, attrs.sum(dim=-1).squeeze().detach().numpy(), pred, delta.item()

print("Helper functions ready.")

## 2. Extract Attention Weights

In [ ]:
def get_attention_weights(text):
    """Extract all attention matrices from BERT.
    
    Returns:
        attentions: list of (1, n_heads, seq_len, seq_len) per layer
        tokens: list of token strings
    """
    ids, mask, ttype = tokenize(text)
    tokens = tokenizer.convert_ids_to_tokens(ids[0].tolist())

    with torch.no_grad():
        outputs = model(
            input_ids=ids,
            attention_mask=mask,
            token_type_ids=ttype,
            output_attentions=True,
        )
    return outputs.attentions, tokens


def compute_mean_attention(attentions, layer: str = 'last'):
    """Compute mean attention from [CLS] to all tokens.
    
    Args:
        layer: 'last' (last layer), 'mean' (average all), or int
    Returns:
        attention_scores: (seq_len,) - attention from [CLS] to each token
    """
    if layer == 'last':
        attn = attentions[-1]  # (1, n_heads, seq_len, seq_len)
    elif layer == 'mean':
        attn = torch.stack(attentions).mean(dim=0)
    else:
        attn = attentions[int(layer)]

    # Average across heads, take [CLS] row (index 0)
    mean_attn = attn.mean(dim=1).squeeze(0)  # (seq_len, seq_len)
    cls_attention = mean_attn[0, :]  # attention from [CLS] to all tokens
    return cls_attention.numpy()


def attention_rollout(attentions):
    """Compute attention rollout: propagate attention through all layers."""
    result = torch.eye(attentions[0].shape[-1])  # identity residual stream
    for layer_attn in attentions:
        mean_attn = layer_attn.mean(dim=1).squeeze(0)  # avg heads
        # Account for residual connections: 50% attention + 50% identity
        with_residual = 0.5 * mean_attn + 0.5 * torch.eye(mean_attn.shape[0])
        # Renormalize rows
        with_residual = with_residual / with_residual.sum(dim=-1, keepdim=True)
        result = with_residual @ result
    return result[0, :].numpy()  # rollout from [CLS]

## 3. Side-by-Side Comparison Function

In [ ]:
def compare_attribution_methods(text, n_steps=40):
    """Compare IG, attention (last layer), and attention rollout for one sentence."""
    # IG
    tokens_all, ig_attrs_all, pred_class, delta = compute_ig_attrs(text, n_steps)

    # Attention
    attentions, _ = get_attention_weights(text)

    # Extract attention scores (skip special tokens for fair comparison)
    attn_last = compute_mean_attention(attentions, layer='last')
    attn_roll = attention_rollout(attentions)

    # Aggregate IG to word level and trim to non-special tokens
    word_tokens, ig_attrs = aggregate_subwords(tokens_all, ig_attrs_all, skip_special=True)

    # For attention: skip [CLS] (index 0) and [SEP] (last token)
    # and aggregate subwords to match word_tokens
    non_special_tokens = [t for t in tokens_all if t not in {'[CLS]', '[SEP]', '[PAD]'}]
    non_special_ig = ig_attrs_all[[i for i, t in enumerate(tokens_all)
                                    if t not in {'[CLS]', '[SEP]', '[PAD]'}]]
    non_special_attn_last = attn_last[[i for i, t in enumerate(tokens_all)
                                        if t not in {'[CLS]', '[SEP]', '[PAD]'}]]
    non_special_attn_roll = attn_roll[[i for i, t in enumerate(tokens_all)
                                        if t not in {'[CLS]', '[SEP]', '[PAD]'}]]

    # Aggregate subword tokens for attention
    _, attn_last_agg = aggregate_subwords(non_special_tokens, non_special_attn_last, skip_special=False)
    _, attn_roll_agg = aggregate_subwords(non_special_tokens, non_special_attn_roll, skip_special=False)

    # Ensure same length
    n = min(len(word_tokens), len(attn_last_agg), len(attn_roll_agg))
    word_tokens = word_tokens[:n]
    ig_attrs    = ig_attrs[:n]
    attn_last_agg = attn_last_agg[:n]
    attn_roll_agg = attn_roll_agg[:n]

    # Spearman correlations
    r_ig_attn, _ = spearmanr(np.abs(ig_attrs), attn_last_agg)
    r_ig_roll, _ = spearmanr(np.abs(ig_attrs), attn_roll_agg)

    return {
        'text': text,
        'words': word_tokens,
        'ig': ig_attrs,
        'attn_last': attn_last_agg,
        'attn_roll': attn_roll_agg,
        'pred_class': pred_class,
        'pred_label': LABELS[pred_class],
        'delta': delta,
        'r_ig_attn': r_ig_attn,
        'r_ig_roll': r_ig_roll,
    }

## 4. Run Comparison on Key Sentences

In [ ]:
comparison_sentences = [
    "The film was absolutely brilliant from start to finish.",       # simple positive
    "The movie was NOT boring at all.",                              # negation
    "A disappointingly average but occasionally engaging film.",     # mixed
    "I did not dislike the film, but I wouldn't call it good.",      # double negation
]

print("Computing comparisons...")
comparisons = []
for text in comparison_sentences:
    comp = compare_attribution_methods(text, n_steps=30)
    comparisons.append(comp)
    print(f"  {text[:45]:<45} | {comp['pred_label']:<8} | r(IG,Attn)={comp['r_ig_attn']:.3f} | r(IG,Roll)={comp['r_ig_roll']:.3f}")

print("\nDone.")

## 5. Visualization: Three-Method Comparison

In [ ]:
def plot_method_comparison(comp, figsize=(16, 6)):
    """Plot IG, attention, and rollout side by side for one sentence."""
    words = comp['words']
    n = len(words)

    methods = [
        ('Integrated Gradients', comp['ig'], True,  '#2ca25f'),    # signed, green/red
        ('Attention (last layer)', comp['attn_last'], False, '#377eb8'),  # unsigned
        ('Attention Rollout', comp['attn_roll'], False, '#984ea3'),       # unsigned
    ]

    fig, axes = plt.subplots(1, 3, figsize=figsize)

    for ax, (title, vals, is_signed, color) in zip(axes, methods):
        ax.axis('off')
        max_abs = np.abs(vals).max() if np.abs(vals).max() > 0 else 1
        vals_norm = vals / max_abs

        for i, (word, v, v_raw) in enumerate(zip(words, vals_norm, vals)):
            x = (i + 0.5) / n
            w = 1.0 / max(n, 1)

            if is_signed:
                if v > 0: r, g, b = 1-v*0.6, 1.0, 1-v*0.6
                else:     r, g, b = 1.0, 1+v*0.6, 1+v*0.6
            else:
                # Intensity by magnitude (blue tones)
                intensity = abs(v)
                r = 1 - intensity * 0.6
                g = 1 - intensity * 0.5
                b = 1.0

            ax.add_patch(plt.Rectangle(
                (x-w*0.44, 0.25), w*0.88, 0.55,
                facecolor=(np.clip(r,0,1), np.clip(g,0,1), np.clip(b,0,1)),
                edgecolor='#dddddd', linewidth=0.4
            ))
            ax.text(x, 0.55, word, ha='center', va='center',
                    fontsize=max(7, 10-n//3), color='black')
            ax.text(x, 0.15, f"{v_raw:.3f}",
                    ha='center', va='center', fontsize=6, color='#666')

        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_title(title, fontsize=9, fontweight='bold', color=color)

    fig.suptitle(
        f'"{comp["text"]}"\n'
        f'Prediction: {comp["pred_label"]} | '
        f'r(IG, Attn)={comp["r_ig_attn"]:.3f} | '
        f'r(IG, Rollout)={comp["r_ig_roll"]:.3f}',
        fontsize=10, fontweight='bold'
    )
    return fig


# Plot all comparisons
for i, comp in enumerate(comparisons):
    fig = plot_method_comparison(comp)
    plt.tight_layout()
    fname = f'attention_vs_ig_case{i+1}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Figure saved: {fname}")

## 6. Negation Case Study

In [ ]:
# Deep dive on negation: compare "boring" vs "not boring"
negation_pairs = [
    ("The movie was boring and dull.", "NEGATIVE"),
    ("The movie was not boring at all.", "POSITIVE"),
]

print("Negation Analysis: 'boring' vs 'not boring'")
print("=" * 55)

fig, axes = plt.subplots(2, 2, figsize=(18, 8))

for row, (text, expected) in enumerate(negation_pairs):
    comp = compare_attribution_methods(text, n_steps=30)
    words = comp['words']

    # Find "boring" and "not" attributions
    boring_ig = boring_attn = not_ig = not_attn = None
    for word, ig_v, attn_v in zip(words, comp['ig'], comp['attn_last']):
        if word.lower() == 'boring':
            boring_ig, boring_attn = ig_v, attn_v
        if word.lower() == 'not':
            not_ig, not_attn = ig_v, attn_v

    print(f"\nText: '{text}'")
    print(f"Prediction: {comp['pred_label']} (expected: {expected})")
    if boring_ig is not None:
        print(f"  'boring' — IG: {boring_ig:+.4f},  Attention: {boring_attn:.4f}")
    if not_ig is not None:
        print(f"  'not'    — IG: {not_ig:+.4f},  Attention: {not_attn:.4f}")

    # Plot IG and attention side by side
    for col, (method_vals, method_title, is_signed) in enumerate([
        (comp['ig'], 'Integrated Gradients', True),
        (comp['attn_last'], 'Attention (last layer)', False)
    ]):
        ax = axes[row, col]
        ax.axis('off')
        n = len(words)
        max_abs = np.abs(method_vals).max() or 1
        vals_norm = method_vals / max_abs

        for i, (word, v, v_raw) in enumerate(zip(words, vals_norm, method_vals)):
            x = (i + 0.5) / n
            w = 1.0 / max(n, 1)
            if is_signed:
                if v > 0: r, g, b = 1-v*0.6, 1.0, 1-v*0.6
                else:     r, g, b = 1.0, 1+v*0.6, 1+v*0.6
            else:
                r, g, b = 1-abs(v)*0.6, 1-abs(v)*0.5, 1.0

            ax.add_patch(plt.Rectangle(
                (x-w*0.44, 0.2), w*0.88, 0.65,
                facecolor=(np.clip(r,0,1), np.clip(g,0,1), np.clip(b,0,1)),
                edgecolor='#ccc', linewidth=0.4
            ))
            ax.text(x, 0.55, word, ha='center', va='center',
                    fontsize=10, color='black',
                    fontweight='bold' if word.lower() in {'not', 'boring'} else 'normal')
            ax.text(x, 0.12, f"{v_raw:.3f}", ha='center', va='center',
                    fontsize=6.5, color='#555')

        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        title_color = '#2ca25f' if comp['pred_label'] == 'POSITIVE' else '#d73027'
        ax.set_title(
            f'{method_title}\n{comp["pred_label"]} | r={comp["r_ig_attn"]:.3f}',
            fontsize=9, color=title_color, fontweight='bold'
        )

plt.suptitle('Negation Analysis: "boring" vs. "NOT boring"\n'
             'Bold = "not" and "boring" tokens. IG vs. Attention disagreement.',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('negation_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: negation_analysis.png")

## 7. Aggregate Correlation Analysis: Many Sentences

In [ ]:
# Compute agreement statistics across a larger set of sentences
evaluation_sentences = [
    "The film was absolutely brilliant from start to finish.",
    "A terrible waste of time and money.",
    "The movie was not boring at all.",
    "An outstanding performance by the lead actor.",
    "I did not dislike the film but I wouldn't recommend it.",
    "A disappointingly average but occasionally engaging film.",
    "The screenplay is tightly written and visually stunning.",
    "Predictable plot and forgettable characters ruin an otherwise decent film.",
    "The direction is surprisingly fresh and inventive.",
    "Not bad, but certainly not memorable either.",
    "A masterfully crafted piece of cinema.",
    "The acting feels stilted and the pacing is slow.",
]

print("Computing correlation statistics across sentences...")
rs_ig_attn = []
rs_ig_roll = []

for text in evaluation_sentences:
    comp = compare_attribution_methods(text, n_steps=20)
    rs_ig_attn.append(comp['r_ig_attn'])
    rs_ig_roll.append(comp['r_ig_roll'])

print(f"\nIG vs. Raw Attention (last layer):")
print(f"  Mean Spearman r: {np.mean(rs_ig_attn):.3f} ± {np.std(rs_ig_attn):.3f}")
print(f"  Range: [{min(rs_ig_attn):.3f}, {max(rs_ig_attn):.3f}]")

print(f"\nIG vs. Attention Rollout:")
print(f"  Mean Spearman r: {np.mean(rs_ig_roll):.3f} ± {np.std(rs_ig_roll):.3f}")
print(f"  Range: [{min(rs_ig_roll):.3f}, {max(rs_ig_roll):.3f}]")

# Distribution plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (corrs, title, color) in zip(axes, [
    (rs_ig_attn, 'IG vs. Attention (last layer)', '#377eb8'),
    (rs_ig_roll, 'IG vs. Attention Rollout', '#984ea3'),
]):
    ax.hist(corrs, bins=10, range=(-1, 1), color=color, alpha=0.7, edgecolor='white')
    ax.axvline(np.mean(corrs), color='black', linestyle='--', linewidth=2,
               label=f'Mean = {np.mean(corrs):.3f}')
    ax.set_xlabel('Spearman rank correlation')
    ax.set_ylabel('Count')
    ax.set_title(f'{title}\nn={len(corrs)} sentences', fontweight='bold')
    ax.legend()
    ax.set_xlim(-1, 1)
    ax.grid(alpha=0.3)

plt.suptitle('Attribution Method Agreement\nIG vs. Attention Methods across 12 Sentences',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('attention_ig_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: attention_ig_correlation.png")

## Self-Check Questions

1. **Negation:** For the sentence "The movie was NOT boring," which method correctly identifies "NOT" as the most attribution-worthy word? What does this tell you about using attention for negation-sensitive tasks?

2. **Correlation interpretation:** The mean Spearman r between IG and attention is somewhere between 0.5 and 0.8 (depending on sentences). Does high correlation mean attention is a reliable proxy for IG? Why or why not?

3. **Rollout vs. raw attention:** Does attention rollout produce higher or lower correlation with IG compared to raw attention? What does rollout add that raw attention lacks?

4. **CLS token:** We skipped [CLS] in the comparison. What does the attention weight on [CLS] represent in BERT's self-attention? Would including it affect the correlation?

**Exercise:** Implement gradient × attention (GradAttn). For each attention weight, multiply it by the absolute gradient of the output with respect to that attention weight. Compare this to raw attention and IG. Does GradAttn produce higher correlation with IG than raw attention?